# APS360 Handwriting Recognition: Step 6 (Model B: CRNN + camera-style augmentation)

**Model A** (Step 2) is the CRNN trained on clean IAM scans. **Model B** is the *same* CRNN with the *same* settings, trained on the same IAM words, except that every training word is randomly distorted to look like a phone/webcam photo. Comparing A and B shows what augmentation does.

### What is data augmentation, and why?
A neural network only learns what it sees. IAM words are clean, flat, evenly lit **scans**. A word seen by a webcam or phone looks different:

| Camera problem | What the augmentation does |
|---|---|
| paper tilted, or seen at an angle | small **rotation**, **perspective warp** and slant (**shear**) |
| shaky hand or moving camera | **motion blur** (or a little focus blur) |
| desk lamp from one side, hand or phone shadow | **uneven lighting** and a soft **shadow** band |
| faded pen, grey paper, cheap camera | lower **contrast**, brightness shift, sensor **noise** |
| phone / webcam video compression | **JPEG** compression |

Each time a training word is used, a random mix of these is applied (each with some probability, at a random strength), so the model almost never sees exactly the same picture twice. The **label stays the same**, so the model learns to read *through* the camera problems. This usually costs a little accuracy on clean scans (the task got harder) but should make the model more robust on real camera images.

### What stays exactly the same as Notebook 2
- the IAM word images, labels (`words_recovered.txt`), filters, skip list / bad-image handling and the **Aachen train/val/test splits**;
- the CRNN architecture and the **hyperparameters of Notebook 2's best run**, read from `outputs/sweep_results.csv` exactly like Step 2 (the sweep is **not** re-run; defaults if there is no CSV);
- `EPOCHS = 100`, early stopping (patience 8), LR halving after 3 epochs without improvement, AdamW, mixed precision, resume from `last.pt`.

### Rules
- Augmentation is applied to the **training split only**. Validation and test are the clean IAM words, so A and B are compared on identical data.
- The webcam test set (`webcam_test/`) is **not used or scored here** (that comes later).
- Model B writes only to its own folders: **`runs/crnn_aug_<settings>/`** (checkpoints) and **`outputs/model_b/`** (results, figures). Notebook 2's `runs/crnn_*` and `outputs/report/` are only *read* (for the A vs B table), never written.

| Section | What happens |
|---|---|
| Setup | GPU check, Drive, code, data (same as Step 2) |
| Augmentation preview | example augmented words, saved as report figures |
| Train Model B | same settings as Model A's best run; re-run to resume |
| Evaluate | CER / WER on **clean** IAM val and test, A vs B table |

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. Run Step 1 once before (so `words_recovered.txt` exists) and ideally Step 2 (so the sweep winner and Model A's numbers exist).

**Time (estimate, not measured on a T4):** ~10 min setup (unzip + load), then roughly **1–1.5 min per epoch** (a bit slower than Step 2 because every word is distorted on the CPU). With early stopping that's usually **~1–2 hours**; the worst case (all 100 epochs) is about 2–2.5 hours. If Colab disconnects, Run all again: it resumes from the last finished epoch.

## Check the GPU
You should see a Tesla T4. If it says CPU, change the runtime type: training on CPU would take many hours.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart and run all.")

## Mount Google Drive
Your Drive appears at `/content/drive/MyDrive`. Allow access when Colab asks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paths
`PROJECT_DIR` is the Drive folder that holds `archive.zip`. If it has been moved, the cell searches your Drive for it.
Heavy files go on Colab's local disk (`/content`), which is far faster than Drive. Results are saved back to Drive.

In [ ]:
import os, subprocess
PROJECT_DIR = "/content/drive/MyDrive/University (Mech Eng)/Year 3/Fall/APS360 (Introduction to Machine Learning)/Project/Project Progress Report"
if not os.path.exists(os.path.join(PROJECT_DIR, "archive.zip")):
    print("archive.zip not found at PROJECT_DIR; searching Drive (can take a minute)...")
    hits = [h for h in subprocess.run(["find", "/content/drive/MyDrive", "-name", "archive.zip", "-size", "+500M"],
                                      capture_output=True, text=True).stdout.split("\n") if h]
    assert hits, "Could not find archive.zip anywhere in your Drive"
    PROJECT_DIR = os.path.dirname(hits[0])

ARCHIVE    = os.path.join(PROJECT_DIR, "archive.zip")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")       # results saved back to Drive
REPORT_DIR = os.path.join(OUTPUT_DIR, "report")         # figures/tables for the progress report
OFFICIAL_WORDS_TXT = os.path.join(PROJECT_DIR, "words.txt")            # full official file, if you add it
RECOVERED_WORDS_TXT = os.path.join(PROJECT_DIR, "words_recovered.txt") # made by the label-recovery step
CODE_DIR   = "/content/aps360_htr"                      # project code (this session only)
RAW_DIR    = "/content/iam_raw"                         # archive.zip unpacked here
DATA_ROOT  = "/content/iam"                             # tidy IAM layout (symlinks into RAW_DIR)
os.environ["HTR_DATA_ROOT"] = DATA_ROOT                 # src/config.py reads this
os.makedirs(REPORT_DIR, exist_ok=True)
MODEL_B_DIR = os.path.join(OUTPUT_DIR, "model_b")          # Model B results (separate from outputs/report)
os.makedirs(MODEL_B_DIR, exist_ok=True)
print("MODEL_B_DIR:", MODEL_B_DIR)
print("PROJECT_DIR:", PROJECT_DIR)

## Write the project code
The new Model B code is shown below as `%%writefile` cells. Everything else (data pipeline, CRNN, sweep helpers, report helpers) is the same code as Steps 1-5 and comes from the embedded zip.

| File | What it does |
|---|---|
| `src/data/augment.py` | the camera-style distortions (`CameraAugment`, `AugConfig`) and the training-only wrapper `AugmentedWords` |
| `src/report/model_b_report.py` | augmentation example figures, training curves, metrics and the Model A vs B table, all in `outputs/model_b/` |
| `src/training/engine.py` | `fit()` from Step 2 (AdamW, mixed precision, LR halving, early stopping, resume) |

### The rest of the codebase (embedded zip)
Every step notebook is generated from one repo; this one embeds code **v16** (`aps360_htr_code_v16.zip` in your project folder).
This cell unpacks the 80 files that aren't shown below (see the other notebook or the zip to read them).
The long string is just that zip, base64-encoded.

In [ ]:
import base64, io, os, shutil, sys, zipfile
# Start from a clean copy every time: delete any older code left in /content/aps360_htr,
# and forget already-imported project modules so this session can't keep using stale code.
os.chdir("/content")
shutil.rmtree("/content/aps360_htr", ignore_errors=True)
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
os.makedirs("/content/aps360_htr", exist_ok=True)
CODE_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).extractall("/content/aps360_htr")
with open("/content/aps360_htr/CODE_VERSION", "w") as f:
    f.write("v16\n")
print("unpacked", len(zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).namelist()), "files | code v16")

In [ ]:
import os
for d in sorted({os.path.dirname(f) for f in ['src/data/augment.py', 'src/report/model_b_report.py', 'src/training/engine.py']}):
    os.makedirs(f'/content/aps360_htr/{d}', exist_ok=True)
%cd /content/aps360_htr

### Camera-style augmentation (new for Model B)
`CameraAugment` applies a random mix of the distortions to a uint8 grayscale word image (same size out). `AugmentedWords` wraps the **training** dataset only: it keeps the training images in RAM at 64 px high, distorts them, then resizes to 32 px with the same preprocessing as Model A. The random choices come from (seed, epoch, word), so the run is reproducible.

In [ ]:
%%writefile /content/aps360_htr/src/data/augment.py
"""Camera-style data augmentation for Model B (training split ONLY).

Model A (Step 2) only ever saw clean scans. A phone or webcam photo of handwriting looks
different: the page is tilted or seen at an angle, the hand shakes (motion blur), lighting is
uneven or a shadow falls across the word, contrast is lower, and the picture is noisy and
JPEG-compressed. Augmentation shows the model random versions of these "camera problems" on the
IAM training words, so it learns to read through them. The label never changes.

Rules this file enforces:
- only the TRAINING split is wrapped (AugmentedWords); validation and test stay clean,
  so Model A and Model B are compared on exactly the same clean IAM words;
- every distortion keeps the image's size and dtype (uint8 grayscale in, same shape out);
- it is reproducible: the random choices for word i in epoch e come from (seed, epoch, i), so
  the same seed gives the same images (also with several DataLoader workers);
- with train=False the wrapper returns the clean (un-augmented) image.
"""
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass

import cv2
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

from src.data.transforms import Preprocess, to_tensor_normalized

cv2.setNumThreads(0)          # DataLoader workers do the parallel work; avoid oversubscription


@dataclass
class AugConfig:
    """Probability and strength of each distortion (one place to change them)."""
    p_perspective: float = 0.5    # seen at an angle: corners moved by up to `perspective` of h
    perspective: float = 0.12
    p_rotate: float = 0.5         # tilt of the paper / camera
    max_rotate_deg: float = 5.0
    p_shear: float = 0.3          # slant change
    max_shear: float = 0.25
    p_motion_blur: float = 0.35   # shaky hand / moving camera
    max_blur_len: int = 7         # pixels at the 64-px working height
    p_gauss_blur: float = 0.2     # out of focus
    p_lighting: float = 0.5       # smooth uneven lighting (brighter on one side)
    p_shadow: float = 0.3         # a soft dark band across the word
    p_contrast: float = 0.5       # faded ink / grey paper
    p_noise: float = 0.3          # sensor noise
    max_noise_std: float = 10.0
    p_jpeg: float = 0.3           # phone / webcam compression
    jpeg_quality: tuple = (25, 80)

    def to_dict(self):
        return {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(self).items()}


def _paper(img):
    """Grey level of the paper (bright background), used to fill borders after a warp."""
    return int(np.percentile(img, 90))


def perspective_warp(img, rng, amount):
    h, w = img.shape
    d = amount * h
    src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
    dst = src + rng.uniform(-d, d, size=(4, 2)).astype(np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,
                               borderValue=_paper(img))


def rotate(img, rng, max_deg):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-max_deg, max_deg), 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,
                          borderValue=_paper(img))


def shear(img, rng, max_shear):
    h, w = img.shape
    s = rng.uniform(-max_shear, max_shear)
    M = np.float32([[1, s, -s * h / 2], [0, 1, 0]])
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,
                          borderValue=_paper(img))


def motion_blur(img, rng, max_len):
    n = int(rng.integers(3, max(4, max_len + 1)))
    k = np.zeros((n, n), np.float32)
    k[n // 2, :] = 1.0
    M = cv2.getRotationMatrix2D(((n - 1) / 2, (n - 1) / 2), rng.uniform(0, 180), 1.0)
    k = cv2.warpAffine(k, M, (n, n))
    k /= max(k.sum(), 1e-6)
    return cv2.filter2D(img, -1, k, borderType=cv2.BORDER_REPLICATE)


def gaussian_blur(img, rng):
    return cv2.GaussianBlur(img, (0, 0), rng.uniform(0.5, 1.4))


def uneven_lighting(img, rng):
    """Multiply by a smooth brightness ramp in a random direction (0.55-1.0 of the brightness)."""
    h, w = img.shape
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    a = rng.uniform(0, 2 * np.pi)
    t = (np.cos(a) * xx / max(w, 1) + np.sin(a) * yy / max(h, 1))
    t = (t - t.min()) / max(t.max() - t.min(), 1e-6)
    lo = rng.uniform(0.55, 0.9)
    return img.astype(np.float32) * (lo + (1 - lo) * t)


def shadow(img, rng):
    """A soft dark band (like the shadow of a hand or phone) across part of the word."""
    h, w = img.shape
    xx = np.arange(w, dtype=np.float32)[None, :]
    yy = np.arange(h, dtype=np.float32)[:, None]
    a = rng.uniform(-0.6, 0.6)                       # band direction (mostly vertical)
    c = rng.uniform(0, w)
    width = rng.uniform(0.15, 0.5) * w
    dist = np.abs(xx - c - a * (yy - h / 2))
    band = np.clip(1 - dist / width, 0, 1) ** 2
    return img.astype(np.float32) * (1 - rng.uniform(0.25, 0.55) * band)


def contrast(img, rng):
    """Lower contrast (faded ink, grey paper) and shift brightness a little."""
    f = img.astype(np.float32)
    m = f.mean()
    return (f - m) * rng.uniform(0.45, 1.0) + m + rng.uniform(-25, 15)


def noise(img, rng, max_std):
    return img.astype(np.float32) + rng.normal(0, rng.uniform(2, max_std), img.shape).astype(np.float32)


def jpeg(img, rng, quality):
    q = int(rng.integers(quality[0], quality[1] + 1))
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, q])
    out = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE) if ok else None
    return out if out is not None and out.shape == img.shape else img


def _u8(x):
    return np.clip(np.rint(x), 0, 255).astype(np.uint8)


class CameraAugment:
    """img (uint8 HxW grayscale) + numpy Generator -> distorted uint8 image of the same shape.
    Geometry first (as the camera sees the page), then optics (blur), then light, then the sensor
    (noise) and finally compression, which is the order they happen in a real photo."""

    def __init__(self, cfg: AugConfig | None = None):
        self.cfg = cfg or AugConfig()

    def __call__(self, img, rng):
        c = self.cfg
        assert img.dtype == np.uint8 and img.ndim == 2, "expects a uint8 grayscale image"
        if min(img.shape) < 4:
            return img.copy()
        x = img
        if rng.random() < c.p_perspective:
            x = perspective_warp(x, rng, c.perspective)
        if rng.random() < c.p_rotate:
            x = rotate(x, rng, c.max_rotate_deg)
        if rng.random() < c.p_shear:
            x = shear(x, rng, c.max_shear)
        if rng.random() < c.p_motion_blur:
            x = motion_blur(x, rng, c.max_blur_len)
        elif rng.random() < c.p_gauss_blur:
            x = gaussian_blur(x, rng)
        if rng.random() < c.p_lighting:
            x = _u8(uneven_lighting(x, rng))
        if rng.random() < c.p_shadow:
            x = _u8(shadow(x, rng))
        if rng.random() < c.p_contrast:
            x = _u8(contrast(x, rng))
        if rng.random() < c.p_noise:
            x = _u8(noise(x, rng, c.max_noise_std))
        if rng.random() < c.p_jpeg:
            x = jpeg(_u8(x), rng, c.jpeg_quality)
        x = _u8(x)
        assert x.shape == img.shape
        return x


class AugmentedWords(Dataset):
    """Wraps the IAM TRAINING dataset for Model B.

    The training images are kept in RAM at `work_height` px (64, twice the model's 32), so the
    distortions are applied at a higher resolution than the model sees; each sample is then
    resized to 32 px and normalised with the SAME Preprocess as training/eval (src/data/transforms.py).
    Same items, labels, skip list and bad-image handling as the wrapped dataset: images that can't be
    decoded are dropped and listed in .dropped (counted by add_dropped()).
    """

    def __init__(self, base, augment=None, seed=0, work_height=64, train=True, num_threads=8, preload=True):
        self.base, self.items, self.charset = base, list(base.items), base.charset
        self.transform = base.transform                      # 32-px resize + normalise (unchanged)
        self.work = Preprocess(height=work_height, min_width=base.transform.min_width * work_height // base.transform.height,
                               max_width=base.transform.max_width * work_height // base.transform.height)
        self.augment = augment or CameraAugment()
        self.seed, self.train = seed, train
        self._epoch = torch.zeros(1, dtype=torch.long).share_memory_()   # seen by DataLoader workers
        self.dropped = list(getattr(base, "dropped", []))
        self._cache = None
        if preload:
            self.preload(num_threads)

    def set_epoch(self, epoch: int):
        self._epoch[0] = int(epoch)

    @property
    def epoch(self):
        return int(self._epoch[0])

    def _load(self, it):
        try:
            with Image.open(it.path) as im:
                img = im.convert("L")
            if img.size[0] == 0 or img.size[1] == 0:
                return None
            return np.asarray(self.work.resize(img), dtype=np.uint8)
        except Exception:
            return None

    def preload(self, num_threads=8):
        with ThreadPoolExecutor(max_workers=num_threads) as pool:
            cache = list(pool.map(self._load, self.items))
        bad = [i for i, a in enumerate(cache) if a is None]
        if bad:
            ids = [self.items[i].word_id for i in bad]
            print(f"dropped {len(ids)} unreadable image(s) while preloading: {', '.join(ids[:10])}")
            self.dropped += ids
            keep = set(range(len(cache))) - set(bad)
            self.items = [it for i, it in enumerate(self.items) if i in keep]
            cache = [a for i, a in enumerate(cache) if i in keep]
        self._cache = cache
        return self

    def __len__(self):
        return len(self.items)

    def widths(self):
        """Width of each sample after the final 32-px resize (for batching similar widths)."""
        return [self.transform.resized_width(a.shape[1], a.shape[0]) for a in self._cache]

    def rng(self, idx, epoch=None):
        return np.random.default_rng([int(self.seed), int(self.epoch if epoch is None else epoch), int(idx)])

    def image(self, idx, epoch=None, augment=None):
        """The uint8 working-resolution image the model input is made from (augmented if train)."""
        a = self._cache[idx]
        if augment if augment is not None else self.train:
            a = self.augment(a, self.rng(idx, epoch))
        return a

    def __getitem__(self, idx):
        it = self.items[idx]
        img = Image.fromarray(self.image(idx))
        x = to_tensor_normalized(self.transform.resize(img))
        target = torch.tensor(self.charset.encode(it.text), dtype=torch.long)
        return x, target, len(target), it.text

### Model B report outputs
Figures and tables saved to `outputs/model_b/`.

In [ ]:
%%writefile /content/aps360_htr/src/report/model_b_report.py
"""Step 6 (Model B = CRNN trained with camera-style augmentation): report figures, tables, numbers.

Everything goes to outputs/model_b/ (its own folder), never to Step 2's outputs/report/.
"""
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.data import augment as A
from src.report import figures, tables
from src.report.style import FULL_WIDTH, save_figure
from src.utils.metrics import cer, wer

pct = lambda x: f"{100 * x:.2f}"

# one distortion at a time, for the "what each augmentation does" figure
SINGLE = [("perspective", lambda im, r: A.perspective_warp(im, r, 0.12)),
          ("rotation", lambda im, r: A.rotate(im, r, 5.0)),
          ("motion blur", lambda im, r: A.motion_blur(im, r, 7)),
          ("uneven light", A.uneven_lighting), ("shadow", A.shadow), ("contrast", A.contrast),
          ("noise", lambda im, r: A.noise(im, r, 10.0)), ("JPEG", lambda im, r: A.jpeg(im, r, (25, 40)))]


def _show(ax, img, title=None):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255, aspect="equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, fontsize=7, pad=2)


def save_augmentation_examples(out_dir, aug_ds, n_words=4, n_aug=4, seed=0):
    """Two figures: (1) a few training words, clean + random augmented versions (exactly what
    Model B trains on, before the 32-px resize); (2) one word with each distortion on its own."""
    rng = np.random.default_rng(seed)
    idx = [int(i) for i in rng.choice(len(aug_ds), size=min(n_words, len(aug_ds)), replace=False)]
    fig, axes = plt.subplots(len(idx), n_aug + 1, figsize=(FULL_WIDTH, 0.55 * len(idx) + 0.3), squeeze=False)
    for r, i in enumerate(idx):
        _show(axes[r][0], aug_ds.image(i, augment=False), "clean (IAM)" if r == 0 else None)
        for k in range(n_aug):
            _show(axes[r][k + 1], aug_ds.image(i, epoch=k + 1, augment=True), f"augmented {k + 1}" if r == 0 else None)
    fig.tight_layout(pad=0.3)
    paths = save_figure(fig, out_dir, "fig_model_b_augmentation_examples")

    img = aug_ds.image(idx[0], augment=False)
    cols = 3
    rows = int(np.ceil((len(SINGLE) + 1) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(FULL_WIDTH, 0.75 * rows + 0.2), squeeze=False)
    flat = [a for row in axes for a in row]
    _show(flat[0], img, "clean")
    for ax, (name, fn) in zip(flat[1:], SINGLE):
        _show(ax, A._u8(fn(img, np.random.default_rng(seed + 1))), name)
    for ax in flat[len(SINGLE) + 1:]:
        ax.axis("off")
    fig.tight_layout(pad=0.3)
    paths += save_figure(fig, out_dir, "fig_model_b_augmentation_types")
    return paths


def save_curves(out_dir, history_csv):
    """Step 2's curve figures, renamed fig_model_b_* so they can't be mixed up with Model A's."""
    out_dir = Path(out_dir)
    paths = figures.plot_training_curves(history_csv, out_dir)
    renamed = []
    for p in paths:
        q = Path(p).with_name(Path(p).name.replace("fig_", "fig_model_b_", 1))
        os.replace(p, q)
        renamed.append(str(q))
    return renamed


def save_metrics(out_dir, preds, eval_results, settings, run_dir, aug_cfg, model_a_metrics=None):
    """metrics.json + table_model_b_metrics (val/test on CLEAN IAM) + table_model_a_vs_b if Model A's
    numbers exist. Returns the numbers dict."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    nums = {"run_dir": str(run_dir), "settings": settings, "augmentation": aug_cfg,
            "note": "CER/WER on clean IAM val/test (no augmentation at evaluation); webcam set not scored here."}
    rows = []
    for split in ("val", "test"):
        p = preds[split]
        refs, hyps = [r["ref"] for r in p], [r["hyp"] for r in p]
        c, w = cer(refs, hyps), wer(refs, hyps)
        nums[split] = {"cer": c, "wer": w, "words": len(p), "loss": eval_results[split]["loss"],
                       "ms_per_word_batched": eval_results[split]["ms_per_word"]}
        rows.append({"Split": split, "Words": len(p), "CER (%)": pct(c), "WER (%)": pct(w)})
    (out_dir / "model_b_metrics.json").write_text(json.dumps(nums, indent=2, default=str))
    paths = tables.write_table(rows, list(rows[0]), out_dir, "table_model_b_metrics",
                               "Model B (CRNN trained with camera-style augmentation) on the clean IAM validation "
                               "and test splits (writer-independent, greedy decoding).", "tab:model_b_metrics")
    if model_a_metrics:
        comp = []
        for name, m in (("Model A (no augmentation)", model_a_metrics), ("Model B (augmentation)", nums)):
            comp.append({"Model": name,
                         "Val CER (%)": pct(m["val"]["cer"]), "Val WER (%)": pct(m["val"]["wer"]),
                         "Test CER (%)": pct(m["test"]["cer"]), "Test WER (%)": pct(m["test"]["wer"])})
        paths += tables.write_table(comp, list(comp[0]), out_dir, "table_model_a_vs_b",
                                    "Same CRNN and hyperparameters, trained without (A) and with (B) camera-style "
                                    "augmentation, both evaluated on clean IAM.", "tab:model_a_vs_b")
        nums["model_a"] = {s: model_a_metrics[s] for s in ("val", "test")}
    return nums, paths


def load_model_a_metrics(run_dir_a):
    """Model A's val/test numbers from Step 2's runs/<RUN_NAME>/metrics.json (read only), or None."""
    p = Path(run_dir_a) / "metrics.json"
    if not p.exists():
        return None
    m = json.loads(p.read_text())
    return m if all(s in m and "cer" in m[s] for s in ("val", "test")) else None

### Training loop (same as Step 2)
Identical to Step 2, plus one line: each epoch it tells the augmented dataset the epoch number, so every epoch gets new random distortions.

In [ ]:
%%writefile /content/aps360_htr/src/training/engine.py
"""Training and evaluation for the CRNN.

The pieces, in the order a training run uses them:
  make_loader       batches of similar width (BucketBatchSampler) + collate_fn
  ctc_loss_on_batch forward pass + CTC loss for one batch
  train_one_epoch   AdamW step per batch, gradient clipping, mixed precision on GPU
  evaluate          greedy decoding -> CER / WER (+ every prediction)
  fit               the epoch loop: LR schedule, best checkpoint, early stopping, resume
"""
import csv
import json
import math
import random
import time
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from src.config import ModelConfig, TrainConfig
from src.data.charset import Charset
from src.data.iam import collate_fn
from src.models.crnn import CRNN, count_parameters
from src.training.sampler import BucketBatchSampler
from src.utils.metrics import cer, wer


def seed_everything(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------------ data

def subset(ds, frac: float, seed: int = 0):
    """Random fraction of a dataset (for quick sweeps). Keeps .widths() working."""
    if frac >= 1.0:
        return ds
    idx = sorted(random.Random(seed).sample(range(len(ds)), max(1, int(len(ds) * frac))))
    sub = Subset(ds, idx)
    sub.widths = lambda: _pick(ds.widths(), idx)
    return sub


def _pick(values, idx):
    return [values[i] for i in idx]


def make_loader(ds, batch_size, shuffle, num_workers=0, seed=0):
    widths = ds.widths()
    sampler = BucketBatchSampler(widths, batch_size, shuffle=shuffle, seed=seed)
    loader = DataLoader(ds, batch_sampler=sampler, collate_fn=collate_fn, num_workers=num_workers,
                        pin_memory=torch.cuda.is_available(), persistent_workers=num_workers > 0)
    return loader, sampler


# ------------------------------------------------------------------ one batch

ctc_loss_fn = nn.CTCLoss(blank=0, reduction="mean", zero_infinity=True)
# zero_infinity: if a label can't fit in the time steps (word too long for its
# width), that sample gives loss 0 instead of inf, so one bad sample can't wreck training.


def ctc_loss_on_batch(model, batch, device, amp=False):
    images = batch["images"].to(device, non_blocking=True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
        log_probs = model(images)                         # (T, B, C)
    input_lengths = CRNN.output_lengths(batch["widths"])  # real time steps per image
    # CTC is computed in float32: it's numerically touchy in half precision.
    loss = ctc_loss_fn(log_probs.float(), batch["targets"].to(device), input_lengths, batch["target_lengths"])
    return loss, log_probs, input_lengths


# ------------------------------------------------------------------ epochs

def train_one_epoch(model, loader, optimizer, scaler, device, cfg: TrainConfig, log_every=100):
    model.train()
    total, n, t0 = 0.0, 0, time.time()
    for i, batch in enumerate(loader):
        if cfg.max_batches and i >= cfg.max_batches:
            break
        if batch is None:                               # every image in it was unreadable
            continue
        loss, _, _ = ctc_loss_on_batch(model, batch, device, amp=scaler.is_enabled())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)                      # so clipping sees real gradient sizes
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * len(batch["texts"]); n += len(batch["texts"])
        if log_every and (i + 1) % log_every == 0:
            print(f"    batch {i+1}/{len(loader)}  loss {total/n:.3f}  ({time.time()-t0:.0f}s)")
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, charset: Charset, device, amp=False, max_batches=None):
    """Greedy-decode a whole split. Returns dict with loss, cer, wer, predictions."""
    model.eval()
    refs, hyps, total, n = [], [], 0.0, 0
    t0 = time.time()
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        if batch is None:                                # every image in it was unreadable
            continue
        loss, log_probs, lengths = ctc_loss_on_batch(model, batch, device, amp=amp)
        best = log_probs.argmax(dim=2).cpu()             # (T, B)
        for b, L in enumerate(lengths.tolist()):
            hyps.append(charset.ctc_greedy_decode(best[:L, b].tolist()))
        refs += batch["texts"]
        total += loss.item() * len(batch["texts"]); n += len(batch["texts"])
    secs = time.time() - t0
    return {"loss": total / max(n, 1), "cer": cer(refs, hyps), "wer": wer(refs, hyps),
            "n": len(refs), "ms_per_word": 1000 * secs / max(len(refs), 1),
            "refs": refs, "hyps": hyps}


# ------------------------------------------------------------------ checkpoints

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_cer, history,
                    model_cfg: ModelConfig, charset: Charset, bad_epochs=0):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "epoch": epoch, "best_cer": best_cer, "history": history, "bad_epochs": bad_epochs,
                "model_cfg": asdict(model_cfg), "charset": charset.chars}, tmp)
    tmp.replace(path)   # write then rename: a Colab disconnect can't leave a half-written file


def load_model(path, device=None):
    """Rebuild the CRNN from a checkpoint. Returns (model, charset, checkpoint dict)."""
    device = device or get_device()
    ck = torch.load(path, map_location=device, weights_only=False)
    mc = ck["model_cfg"]; mc["cnn_channels"] = tuple(mc["cnn_channels"])
    charset = Charset(ck["charset"])
    model = CRNN(charset.num_classes, ModelConfig(**mc)).to(device)
    model.load_state_dict(ck["model"])
    model.eval()
    return model, charset, ck


# ------------------------------------------------------------------ the full run

def fit(datasets, charset: Charset, model_cfg: ModelConfig, cfg: TrainConfig, resume=True, verbose=True):
    """Train on datasets['train'], pick the best epoch by val CER.

    Saves to cfg.out_dir:  best.pt (lowest val CER), last.pt (for resuming),
    history.csv (one row per epoch) and config.json.
    """
    seed_everything(cfg.seed)
    device = get_device()
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    amp = cfg.amp and device.type == "cuda"

    train_ds = subset(datasets["train"], cfg.train_frac, cfg.seed)
    train_loader, train_sampler = make_loader(train_ds, cfg.batch_size, True, cfg.num_workers, cfg.seed)
    val_loader, _ = make_loader(datasets["val"], cfg.batch_size * 2, False, cfg.num_workers)

    model = CRNN(charset.num_classes, model_cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    # Halve the LR when val CER stops improving for lr_patience epochs.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
                                                           patience=cfg.lr_patience)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    start, best_cer, history, bad = 1, math.inf, [], 0

    last = out / "last.pt"
    if resume and last.exists():
        ck = torch.load(last, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
        start, best_cer, history = ck["epoch"] + 1, ck["best_cer"], ck["history"]
        bad = ck.get("bad_epochs", 0)
        print(f"Resuming from epoch {ck['epoch']} (best val CER so far {best_cer:.4f})")

    (out / "config.json").write_text(json.dumps(
        {"model": asdict(model_cfg), "train": {k: str(v) for k, v in asdict(cfg).items()},
         "train_words": len(train_ds), "val_words": len(datasets["val"]),
         "parameters": count_parameters(model), "device": str(device)}, indent=2))
    if verbose:
        print(f"device {device} | {count_parameters(model):,} parameters | "
              f"{len(train_ds)} train / {len(datasets['val'])} val words | {len(train_loader)} batches/epoch")

    for epoch in range(start, cfg.epochs + 1):
        if bad >= cfg.patience:
            print(f"Early stop: val CER hasn't improved for {cfg.patience} epochs"); break
        t0 = time.time()
        train_sampler.set_epoch(epoch)
        if hasattr(train_ds, "set_epoch"):          # Model B: new random augmentations each epoch
            train_ds.set_epoch(epoch)
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device, cfg,
                                     log_every=100 if verbose else 0)
        val = evaluate(model, val_loader, charset, device, amp=amp, max_batches=cfg.max_batches)
        scheduler.step(val["cer"])
        improved = val["cer"] < best_cer
        if improved:
            best_cer, bad = val["cer"], 0
        else:
            bad += 1
        row = {"epoch": epoch, "train_loss": round(train_loss, 4), "val_loss": round(val["loss"], 4),
               "val_cer": round(val["cer"], 4), "val_wer": round(val["wer"], 4),
               "lr": optimizer.param_groups[0]["lr"], "secs": round(time.time() - t0, 1)}
        history.append(row)
        if improved:
            save_checkpoint(out / "best.pt", model, optimizer, scheduler, scaler, epoch, best_cer,
                            history, model_cfg, charset, bad)
        save_checkpoint(last, model, optimizer, scheduler, scaler, epoch, best_cer, history,
                        model_cfg, charset, bad)
        with open(out / "history.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerows(history)
        if verbose:
            k = max(1, len(val["refs"]) // 4)          # 3 examples spread across the val set
            ex = ", ".join(f"{r!r}->{h!r}" for r, h in list(zip(val["refs"], val["hyps"]))[k::k][:3])
            print(f"epoch {epoch:3d} | train loss {train_loss:.3f} | val loss {val['loss']:.3f} | "
                  f"val CER {val['cer']:.4f} WER {val['wer']:.4f} | lr {row['lr']:.1e} | "
                  f"{row['secs']:.0f}s {'*best*' if improved else ''}\n          e.g. {ex}")
    return {"best_val_cer": best_cer, "history": history, "out_dir": str(out)}


def overfit_one_batch(dataset, charset: Charset, model_cfg=None, steps=200, batch_size=16, lr=1e-3):
    """Sanity check: a correct model + loss should memorise one small batch.
    Returns the list of losses (should fall towards 0) and the final predictions."""
    seed_everything(0)
    device = get_device()
    batch = collate_fn([dataset[i] for i in range(min(batch_size, len(dataset)))])
    model = CRNN(charset.num_classes, model_cfg or ModelConfig(dropout=0.0)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    model.train()
    for _ in range(steps):
        loss, _, _ = ctc_loss_on_batch(model, batch, device)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        _, log_probs, lengths = ctc_loss_on_batch(model, batch, device)
    best = log_probs.argmax(2).cpu()
    preds = [charset.ctc_greedy_decode(best[:L, b].tolist()) for b, L in enumerate(lengths.tolist())]
    return losses, list(zip(batch["texts"], preds))

## Install missing packages
Colab already has PyTorch, OpenCV, NumPy, Pillow, matplotlib, pyarrow and pytest, so this only installs what's missing.
`editdistance` (faster CER/WER) is optional; the code falls back to pure Python without it.

In [ ]:
import importlib.util, subprocess, sys
need = {"torch": "torch", "torchvision": "torchvision", "cv2": "opencv-python", "numpy": "numpy",
        "PIL": "pillow", "matplotlib": "matplotlib", "tqdm": "tqdm", "pytest": "pytest",
        "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
        "pytesseract": "pytesseract", "editdistance": "editdistance"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
print("missing:", missing or "nothing")
for pkg in missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    note = " (optional: CER/WER use the pure-Python fallback)" if pkg == "editdistance" else ""
    print(f"  {pkg}: {'installed' if r.returncode == 0 else 'install failed' + note}")

## Unpack archive.zip (about 1.2 GB)
Copied to Colab's disk and unzipped there, which takes a few minutes. Re-running in the same session skips it.

In [ ]:
import os, shutil, time
marker = os.path.join(RAW_DIR, ".unzipped")
if os.path.exists(marker):
    print("Already unzipped this session:", RAW_DIR)
else:
    t = time.time()
    shutil.copy(ARCHIVE, "/content/archive.zip")
    print(f"copied in {time.time()-t:.0f}s"); t = time.time()
    !unzip -q -o /content/archive.zip -d {RAW_DIR}
    os.remove("/content/archive.zip")
    open(marker, "w").close()
    print(f"unzipped in {time.time()-t:.0f}s")
!ls {RAW_DIR}

## Find words.txt and the images
`setup_iam.py` finds the word images and a `words.txt` in any download layout (official tgz, Kaggle `iam_words/`, or a flat folder) and links them into `/content/iam`.

**Which labels get used** (first one that exists wins):
1. `words.txt` in your project folder: the full official file, if you download it from IAM;
2. `words_recovered.txt` in your project folder, made by the label-recovery step below;
3. the `words.txt` inside `archive.zip`. The Kaggle copy is **cut off at e07-072**, so only ~39% of IAM has labels.

In [ ]:
def pick_words_txt():
    for p in (OFFICIAL_WORDS_TXT, RECOVERED_WORDS_TXT):
        if os.path.exists(p):
            return p
    return None

def run_setup():
    wt = pick_words_txt()
    print("labels from:", wt or "the words.txt inside archive.zip")
    extra = f'--words-txt "{wt}"' if wt else ""
    !python scripts/setup_iam.py --raw {RAW_DIR} --out {DATA_ROOT} {extra}

run_setup()

## Recover the missing labels (only needed once)
**Why:** the Kaggle `words.txt` stops part-way through line `e07-072-09-`, but `archive.zip` has the images for all of IAM. Without the missing lines, about 60% of the images can't be used.

**How:** every original IAM PNG has its form's XML stored *inside the file* (a PNG text chunk). That XML gives the word's ID, its text, its tag, its bounding box and its line's segmentation status (ok/err), which are exactly the fields of a `words.txt` line. `recover_labels.py`:
1. first tries the XML inside **your own** images (fast, no download);
2. if they've been stripped, it downloads the Hugging Face copy of IAM ([priyank-m/IAM_words_text_recognition](https://huggingface.co/datasets/priyank-m/IAM_words_text_recognition), ~1.2 GB), whose PNGs still carry the XML, and matches **by word ID**. Images whose ID can't be found fall back to a pixel-hash match.

**Checks it prints, for the report:**
- *rebuilt vs original*: for words that have both, the rebuilt line must equal the original Kaggle line (expect ~100%);
- *pixel check*: the HF image is pixel-identical to yours for a sample of recovered words;
- matched / unmatched / ambiguous counts. Below 90%, nothing is written and training uses the labelled subset.

The result is saved to your project folder as `words_recovered.txt` (original lines first, then a `#--- recovered entries below ---` section), so later sessions and the Step 2 notebook reuse it. Recovered words keep their real IAM IDs, so the **Aachen writer-disjoint splits still apply**.

In [ ]:
if os.path.exists(OFFICIAL_WORDS_TXT):
    print("Using the official full words.txt; no recovery needed.")
elif os.path.exists(RECOVERED_WORDS_TXT):
    print("Reusing", RECOVERED_WORDS_TXT)
else:
    !python scripts/recover_labels.py --data-root {DATA_ROOT} --out "{RECOVERED_WORDS_TXT}" \
        --hf-cache /content/hf_iam --report "{OUTPUT_DIR}/label_recovery.json" \
        --preview "{OUTPUT_DIR}/label_recovery_preview.png"
    if os.path.exists(RECOVERED_WORDS_TXT):
        run_setup()          # re-link with the recovered labels
        from IPython.display import Image as _I, display
        display(_I(os.path.join(OUTPUT_DIR, "label_recovery_preview.png")))

## Run the tests
The augmentation tests (same shape and type out, only applied in training mode, same images for the same seed, val/test stay clean) plus the model, data and training tests from Step 2. Every line should say `passed`.

In [ ]:
!python -m pytest -q tests/test_augment.py tests/test_training.py tests/test_model.py tests/test_data.py

## Load the data (same as Step 2)
Same `load_data()` as Step 2: same filters, skip list (`a01-117-05-02`, `r06-022-03-05`, ...), Aachen splits and bad-image handling. Validation is preloaded at 32 px exactly like Step 2.

The **training** words are then wrapped by `AugmentedWords`, which preloads them at 64 px (twice the model's height) so the distortions look natural, and distorts them on the fly. That takes a few minutes and about 1 GB of RAM. Any image that can't be decoded is dropped and counted as `bad_or_missing_image`, like in Step 2.

In [ ]:
from src.training.data import load_data, add_dropped
from src.data.augment import AugConfig, AugmentedWords, CameraAugment
AUG_SEED = 42
AUG_CFG = AugConfig()          # probabilities and strengths of each distortion (see the code above)
datasets, charset, stats = load_data(DATA_ROOT, splits=("val",), preload=True, check_images=False)
import time; t = time.time()
train_b = AugmentedWords(datasets["train"], CameraAugment(AUG_CFG), seed=AUG_SEED)
add_dropped(stats, {"train": train_b})
print(f"training words (augmented): {len(train_b)} | val words (clean): {len(datasets['val'])} | {time.time() - t:.0f}s")
print({k: stats[k] for k in ("kept", "segmentation_err", "bad_or_missing_image", "malformed_line", "split_method")})
assert not isinstance(datasets["val"], AugmentedWords) and datasets["val"].transform.augment is None

## Augmentation preview (figures for the report)
- `fig_model_b_augmentation_examples`: four training words, clean and with four random augmentations (exactly what Model B trains on);
- `fig_model_b_augmentation_types`: one word with each distortion on its own.

Saved to `outputs/model_b/` as PNG (300 dpi) + PDF. If the words look unreadable to *you*, the augmentation is too strong: lower the numbers in `AugConfig` and re-run (that changes the training, so use it before training starts).

In [ ]:
from IPython.display import Image, display
from src.report import model_b_report as mb
for p in mb.save_augmentation_examples(MODEL_B_DIR, train_b, seed=AUG_SEED):
    if p.endswith(".png"):
        display(Image(p, width=700))

## Train Model B
**Settings:** the same choice as Step 2's Train cell: the best row of `outputs/sweep_results.csv` (lowest val CER, ties by val WER), or the defaults (lr 1e-3, hidden 256, dropout 0.3, batch 64, 2 LSTM layers) if there's no usable CSV. The sweep is **not** re-run.

- `EPOCHS = 100`, early stopping after 8 epochs without a val-CER improvement, LR halves after 3, AdamW, mixed precision: all as in Step 2.
- Checkpoints go to **`runs/crnn_aug_<settings>/`** (e.g. `runs/crnn_aug_lr0.001_h256_d0.3_b32_L2`): `best.pt`, `last.pt`, `history.csv`, `config.json`, `augmentation.json`. That name can never be the same as Model A's `runs/crnn_<settings>/`.
- **If Colab disconnects:** Run all again; this cell resumes from `last.pt`. It refuses to resume if the settings or the augmentation changed (then it tells you why).
- Validation each epoch uses the **clean** val words.
- Two DataLoader workers distort the next batches on the CPU while the GPU trains.

In [ ]:
import json
from src.training.engine import fit
from src.training.sweep import choose_settings, make_configs, check_run_dir, run_name
EPOCHS = 100
SWEEP_CSV = os.path.join(OUTPUT_DIR, "sweep_results.csv")      # read only (Step 2's sweep)
SETTINGS, SETTINGS_INFO = choose_settings(SWEEP_CSV)
print(f"Settings from: {SETTINGS_INFO['source'].upper()} - {SETTINGS_INFO['reason']}")
for k, v in SETTINGS.items():
    print(f"  {k:12s} = {v:g}" if isinstance(v, float) else f"  {k:12s} = {v}")

RUN_NAME_A = "crnn_" + run_name(SETTINGS)                    # Model A (Step 2), only read
RUN_NAME   = "crnn_aug_" + run_name(SETTINGS)                # Model B
RUN_DIR    = os.path.join(PROJECT_DIR, "runs", RUN_NAME)
assert RUN_NAME != RUN_NAME_A and os.path.basename(RUN_DIR).startswith("crnn_aug_")
os.makedirs(RUN_DIR, exist_ok=True)
model_cfg, train_cfg = make_configs(SETTINGS, epochs=EPOCHS, patience=8, num_workers=2, out_dir=RUN_DIR)
print("RUN_DIR:", RUN_DIR, "|", check_run_dir(RUN_DIR, model_cfg, train_cfg))

aug_info = {"seed": AUG_SEED, "config": AUG_CFG.to_dict(), "work_height": 64}
aug_json = os.path.join(RUN_DIR, "augmentation.json")
if os.path.exists(aug_json) and json.load(open(aug_json)) != aug_info:
    raise RuntimeError(f"{RUN_DIR} was trained with a different augmentation, so it can't be resumed.\n"
                       f"saved: {json.load(open(aug_json))}\nnow:   {aug_info}\n"
                       "Put AugConfig back as it was, or delete that run folder to start over.")
json.dump(aug_info, open(aug_json, "w"), indent=2)
json.dump({"settings": SETTINGS, **SETTINGS_INFO}, open(os.path.join(RUN_DIR, "settings_choice.json"), "w"),
          indent=2, default=str)
result = fit({"train": train_b, "val": datasets["val"]}, charset, model_cfg, train_cfg, resume=True)
print("best val CER (clean val):", result["best_val_cer"])

## Training curves
Saved as `fig_model_b_training_curves`, `fig_model_b_loss_curve` and `fig_model_b_val_cer_wer_curve` in `outputs/model_b/`. Train loss is on *augmented* words and val loss on *clean* words, so train loss can sit above val loss: that's normal here.

In [ ]:
for p in mb.save_curves(MODEL_B_DIR, os.path.join(RUN_DIR, "history.csv")):
    if p.endswith(".png"):
        display(Image(p, width=700))

## Evaluate Model B on clean IAM val and test
`best.pt` is reloaded and greedy-decoded on the **clean** validation and test words (unseen writers, Aachen split), exactly like Step 2. If Model A's `runs/crnn_<settings>/metrics.json` exists (from Step 2), the **Model A vs Model B** table is made too.

Saved to `outputs/model_b/`: `model_b_metrics.json`, `table_model_b_metrics` (CSV + LaTeX), `table_model_a_vs_b` (CSV + LaTeX), `model_b_val_predictions.csv`, `model_b_test_predictions.csv`, and `crnn_model_b.pt` (small CPU copy of the weights, like the live demo's `crnn_model.pt`, for scoring the webcam set later).

In [ ]:
import torch
from IPython.display import Markdown
from src.training.engine import evaluate, get_device, load_model, make_loader
from src.report.predict import predict_dataset, save_predictions
from src.report.tables import csv_to_markdown
from src.demo.webcam import export_model
device = get_device()
model, cs, ck = load_model(os.path.join(RUN_DIR, "best.pt"), device)
print(f"Model B best epoch {ck['epoch']}, val CER {ck['best_cer']:.4f}")
test_sets, _, _ = load_data(DATA_ROOT, splits=("test",), preload=True, check_images=False)
add_dropped(stats, test_sets)
eval_sets = {"val": datasets["val"], "test": test_sets["test"]}     # clean, never augmented
results, preds = {}, {}
for split, ds in eval_sets.items():
    loader, _ = make_loader(ds, 128, shuffle=False)
    results[split] = evaluate(model, loader, cs, device, amp=device.type == "cuda")
    preds[split] = predict_dataset(model, ds, cs, device, batch_size=128, amp=device.type == "cuda")
    save_predictions(preds[split], os.path.join(MODEL_B_DIR, f"model_b_{split}_predictions.csv"))
    r = results[split]
    print(f"{split:4s}: {r['n']} words | CER {r['cer']:.4f} | WER {r['wer']:.4f}")
json.dump({s: {k: results[s][k] for k in ("cer", "wer", "n", "ms_per_word")} for s in results},
          open(os.path.join(RUN_DIR, "metrics.json"), "w"), indent=2)

model_a = mb.load_model_a_metrics(os.path.join(PROJECT_DIR, "runs", RUN_NAME_A))
print("Model A metrics:", "found in runs/" + RUN_NAME_A if model_a else "not found (run Step 2's Evaluate cell); no A vs B table")
nums, paths = mb.save_metrics(MODEL_B_DIR, preds, results, SETTINGS, RUN_DIR, aug_info, model_a)
for p in paths:
    if p.endswith(".csv"):
        display(Markdown(csv_to_markdown(p)))
export_model(os.path.join(RUN_DIR, "best.pt"), os.path.join(MODEL_B_DIR, "crnn_model_b.pt"),
             extra={"model": "B (camera-style augmentation)", "run": RUN_NAME})
print("saved crnn_model_b.pt")

In [ ]:
print("Files in outputs/model_b:")
for f in sorted(os.listdir(MODEL_B_DIR)):
    print(f"  {os.path.getsize(os.path.join(MODEL_B_DIR, f)) / 1024:8.1f} KB  {f}")

## Next
- **Webcam scoring is in Step 5:** once `webcam_test/` is collected, re-run Step 5; it scores Model A (`live_demo/crnn_model.pt`), Model B (`outputs/model_b/crnn_model_b.pt`, saved here), Tesseract and EMNIST on the same words. That's where augmentation should help most.
- For the report: `fig_model_b_augmentation_examples` (what augmentation looks like), `table_model_a_vs_b` (clean IAM: what it costs or gains), training curves.